# Class Activity: MySQL Installation, Workbench Connection, and Product CRUD

## Purpose

In this activity, you will confirm that MySQL is installed correctly, connect to it using MySQL Workbench, install the Python connector, and perform full CRUD operations using Python in Jupyter Notebook.

CRUD means:

- **Create**: insert new records
- **Read**: retrieve records
- **Update**: modify existing records
- **Delete**: remove records

You will create a database, create a table named `products`, insert a few products, read them, update them, delete them, and verify the final result.


## Part 1: Install MySQL Server

Complete this before running the Python code.

### Student Checklist

1. Download and install **MySQL Community Server**.
2. During installation, define a password for the `root` account.
3. Write your password somewhere safe. You will need it in this notebook.
4. Confirm that the MySQL Server service is running.

### Important

Your password is private. Do not share it with classmates, screenshots, or submissions.


## Part 2: Install MySQL Workbench

Complete this before running the Python code.

### Student Checklist

1. Download and install **MySQL Workbench**.
2. Open MySQL Workbench.
3. Create or open a local connection.
4. Use:
   - Host: `localhost`
   - User: `root`
   - Password: the password you defined during MySQL installation
5. Confirm that Workbench connects successfully.

### Evidence of Completion

Before continuing, you should be able to open MySQL Workbench and connect to your local MySQL Server.


## Part 3: Install the MySQL Connector for Python

The connector allows Python to communicate with MySQL.


In [ ]:
# Run this cell once.
# If it is already installed, you may see a message saying the requirement is already satisfied.

!pip install mysql-connector-python

## Part 4: Import Required Library

In [ ]:
import mysql.connector
from mysql.connector import Error

## Part 5: Define Your MySQL Credentials

Replace `YOUR_PASSWORD_HERE` with the password you created for the MySQL `root` account.

Do not add extra spaces before or after the password.


In [ ]:
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = "YOUR_PASSWORD_HERE"  # TODO: Replace with your own root password
DATABASE_NAME = "college_activity" 

## Part 6: Connect to the MySQL Server

This connection is made to the MySQL Server first, before selecting a specific database.


In [ ]:
def connect_to_mysql_server():
    """Connect to the local MySQL Server without selecting a database yet."""
    try:
        connection = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD
        )

        if connection.is_connected():
            print("Successfully connected to MySQL Server.")
            return connection

    except Error as e:
        print("Connection failed.")
        print("Error:", e)
        return None


server_connection = connect_to_mysql_server()

## Part 7: Create a Database

We will create a database named `college_activity`.

`IF NOT EXISTS` prevents an error if the database already exists.


In [ ]:
def create_database(connection, database_name):
    """Create a database if it does not already exist."""
    cursor = connection.cursor()
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {database_name}")
    cursor.close()
    print(f"Database '{database_name}' is ready.")


# TODO: Call the function to create the database.
# create_database(server_connection, DATABASE_NAME)

## Part 8: Connect to the Specific Database

After creating the database, we connect directly to it.


In [ ]:
def connect_to_database():
    """Connect directly to the selected MySQL database."""
    try:
        connection = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD,
            database=DATABASE_NAME
        )

        if connection.is_connected():
            print(f"Successfully connected to database '{DATABASE_NAME}'.")
            return connection

    except Error as e:
        print("Database connection failed.")
        print("Error:", e)
        return None


# TODO:
# 1. Close the server-level connection if it exists.
# 2. Connect to the database and store the connection in db_connection.

# if server_connection:
#     server_connection.close()

# db_connection = connect_to_database()

## Part 9: Create the `products` Table

The table will contain:

| Column | Type | Description |
|---|---|---|
| product_id | INT | Primary key, generated automatically |
| product_name | VARCHAR(100) | Product name |
| category | VARCHAR(100) | Product category |
| price | DECIMAL(10,2) | Product price |
| quantity | INT | Number of units available |


In [ ]:
def create_products_table(connection):
    """Create the products table if it does not already exist."""
    cursor = connection.cursor()

    create_table_query = """
    CREATE TABLE IF NOT EXISTS products (
        product_id INT AUTO_INCREMENT PRIMARY KEY,
        product_name VARCHAR(100) NOT NULL,
        category VARCHAR(100),
        price DECIMAL(10, 2) NOT NULL,
        quantity INT NOT NULL
    )
    """

    cursor.execute(create_table_query)
    cursor.close()
    print("Table 'products' is ready.")


# TODO: Call the function to create the products table.
# create_products_table(db_connection)

## Part 10: CREATE Operation

Insert two or three products into the `products` table.

The `%s` placeholders help safely pass values into the SQL statement.


In [ ]:
def insert_product(connection, product_name, category, price, quantity):
    """Insert one product into the products table."""
    cursor = connection.cursor()

    insert_query = """
    INSERT INTO products (product_name, category, price, quantity)
    VALUES (%s, %s, %s, %s)
    """

    product_data = (product_name, category, price, quantity)

    cursor.execute(insert_query, product_data)
    connection.commit()

    print(f"Inserted product: {product_name}")

    cursor.close()


# TODO: Insert at least three products.
# Example:
# insert_product(db_connection, "Laptop", "Electronics", 999.99, 10)

## Part 11: READ Operation

Retrieve and display all products from the table.


In [ ]:
def read_products(connection):
    """Read and display all products."""
    cursor = connection.cursor()

    select_query = """
    SELECT product_id, product_name, category, price, quantity
    FROM products
    ORDER BY product_id
    """

    cursor.execute(select_query)
    rows = cursor.fetchall()

    if len(rows) == 0:
        print("No products found.")
    else:
        print("Current products:")
        for row in rows:
            print(row)

    cursor.close()


# TODO: Read all products.
# read_products(db_connection)

## Part 12: UPDATE Operation

Update the price and quantity of one product.

You need to know the `product_id` of the product you want to update. Use the READ operation first to find it.


In [ ]:
def update_product(connection, product_id, new_price, new_quantity):
    """Update the price and quantity for one product."""
    cursor = connection.cursor()

    update_query = """
    UPDATE products
    SET price = %s, quantity = %s
    WHERE product_id = %s
    """

    cursor.execute(update_query, (new_price, new_quantity, product_id))
    connection.commit()

    print(f"Updated product with ID: {product_id}")

    cursor.close()


# TODO:
# 1. Choose an existing product_id.
# 2. Update that product.
# 3. Read the table again to verify the change.

# update_product(db_connection, 1, 899.99, 8)
# read_products(db_connection)

## Part 13: DELETE Operation

Delete one product from the table.

Use the READ operation first to confirm the `product_id`.


In [ ]:
def delete_product(connection, product_id):
    """Delete one product using its product_id."""
    cursor = connection.cursor()

    delete_query = """
    DELETE FROM products
    WHERE product_id = %s
    """

    cursor.execute(delete_query, (product_id,))
    connection.commit()

    print(f"Deleted product with ID: {product_id}")

    cursor.close()


# TODO:
# 1. Choose an existing product_id.
# 2. Delete that product.
# 3. Read the table again to verify the deletion.

# delete_product(db_connection, 2)
# read_products(db_connection)

## Part 14: Full CRUD Practice Task

Complete the following sequence independently:

1. Insert a new product of your choice.
2. Read all products.
3. Update the product you inserted.
4. Read all products again.
5. Delete the product you inserted.
6. Read all products one final time.

Use the helper functions above.


In [ ]:
# TODO: Complete your full CRUD practice here.

# 1. Create

# 2. Read

# 3. Update

# 4. Read again

# 5. Delete

# 6. Final read


## Part 15: Optional Cleanup

Use this only if you want to remove all rows from the table and restart the activity.

Warning: this deletes all product records.


In [ ]:
def delete_all_products(connection):
    """Delete all records from the products table."""
    cursor = connection.cursor()
    cursor.execute("DELETE FROM products")
    connection.commit()
    cursor.close()
    print("All products were deleted.")


# Optional:
# delete_all_products(db_connection)
# read_products(db_connection)

## Part 16: Close the Database Connection

Always close the connection when you finish working with the database.


In [ ]:
# Run this cell at the end of the activity.

# if db_connection and db_connection.is_connected():
#     db_connection.close()
#     print("Database connection closed.")

## Submission / Completion Checklist

By the end of this activity, you should be able to confirm that:

- MySQL Server is installed.
- MySQL Workbench is installed.
- You can connect to MySQL using the `root` account.
- `mysql-connector-python` is installed.
- You created a database.
- You created the `products` table.
- You inserted products.
- You read products.
- You updated at least one product.
- You deleted at least one product.
- You completed the full CRUD cycle.
